# Environment Setup & Reproducibility Initialization

## tujuan notebook
notebook ini menyiapkan environment Google Colab yang konsisten dan reproducible untuk seluruh proyek. fokus pada:
- verifikasi ketersediaan GPU
- instalasi dependencies utama (transformers, datasets, dll)
- konfigurasi reproducibility (random seed, deterministic operations)
- setup struktur direktori proyek
- definisi helper functions dasar yang akan digunakan di notebook lain

**catatan**: notebook ini tidak melakukan training atau eksperimen apapun, hanya setup environment.

## input yang dibutuhkan
- **tidak ada input eksternal** - notebook ini self-contained
- **persyaratan**:
  - google Colab dengan GPU aktif (Runtime > Change runtime type > GPU)
  - akses internet untuk download dependencies

## output yang dihasilkan
- environment variables:
  - `SEED = 42` (untuk reproducibility)
  - `DEVICE` (cuda/cpu)
  - `DTYPE` (float16/float32)
- struktur direktori di `/content/drive/MyDrive/opinion_project/`:
  - `models/sentiment/`
  - `models/summarization/`
  - `outputs/`
  - `cache/`
- helper function: `load_sentiment_model()` (untuk digunakan di notebook lain)

## dependencies dari notebook lain
- **tidak ada** - notebook ini adalah entry point pertama
- **digunakan oleh**: semua notebook berikutnya (01-05) mengasumsikan environment ini sudah disetup

## urutan eksekusi
**Notebook ini HARUS dijalankan pertama kali** sebelum notebook lainnya.

### Cek Runtime & GPU

In [ ]:
# cek apakah GPU tersedia (wajib untuk training)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU di Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


### Install Dependencies

In [ ]:
# install library utama NLP
!pip install -q transformers datasets evaluate accelerate sentencepiece gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


### Import & Versi Library

In [ ]:
import torch
import random
import numpy as np
import transformers
import datasets

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

torch: 2.9.0+cu126
transformers: 5.0.0
datasets: 4.0.0


### Set Random Seed (Reproducibility)

In [ ]:
# semua randomness dikunci agar hasil konsisten
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


### Konfigurasi Device & Precision

In [ ]:
# device abstraction (dipakai di semua notebook)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)

DEVICE: cuda
DTYPE: torch.float16


### Struktur Direktori Proyek

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/opinion_project"

DIRS = {
    "models": f"{BASE_DIR}/models",
    "sentiment": f"{BASE_DIR}/models/sentiment",
    "summarization": f"{BASE_DIR}/models/summarization",
    "outputs": f"{BASE_DIR}/outputs",
    "cache": f"{BASE_DIR}/cache",
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

DIRS

Mounted at /content/drive


{'models': '/content/drive/MyDrive/opinion_project/models',
 'sentiment': '/content/drive/MyDrive/opinion_project/models/sentiment',
 'summarization': '/content/drive/MyDrive/opinion_project/models/summarization',
 'outputs': '/content/drive/MyDrive/opinion_project/outputs',
 'cache': '/content/drive/MyDrive/opinion_project/cache'}

### Helper Function Dasar

In [ ]:
# helper umum agar tidak duplikasi di notebook lain
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def load_sentiment_model(model_name, num_labels=2):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    return tokenizer, model.to(DEVICE)